In [1]:
import requests
import os
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timedelta
import time
import numpy as np
load_dotenv()
API_KEY = os.getenv("NOAA_API_KEY")

In [2]:
df = pd.read_csv(r"D:\HK242\flight_n_delay\flights_sample_3m.csv")
df.head()

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,0.0,181.0,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN


In [ ]:
unique_origins = df[["ORIGIN", "ORIGIN_CITY"]].drop_duplicates()
unique_dest = df[["DEST", "DEST_CITY"]].drop_duplicates()

# Tìm các sân bay chỉ có ở DEST nhưng không có trong ORIGIN
only_in_dest = unique_dest[~unique_dest["DEST"].isin(unique_origins["ORIGIN"])]

# Tìm các sân bay chỉ có ở ORIGIN nhưng không có trong DEST
only_in_origin = unique_origins[~unique_origins["ORIGIN"].isin(unique_dest["DEST"])]

print("Số sân bay chỉ xuất hiện trong DEST nhưng không có ở ORIGIN:", len(only_in_dest))
print(only_in_dest.head())

print("Số sân bay chỉ xuất hiện trong ORIGIN nhưng không có ở DEST:", len(only_in_origin))
print(only_in_origin.head())


Số sân bay chỉ xuất hiện trong DEST nhưng không có ở ORIGIN: 0
Empty DataFrame
Columns: [DEST, DEST_CITY]
Index: []
Số sân bay chỉ xuất hiện trong ORIGIN nhưng không có ở DEST: 0
Empty DataFrame
Columns: [ORIGIN, ORIGIN_CITY]
Index: []


In [28]:
state_fips = {
    "AL": "01", "AK": "02", "AZ": "04", "AR": "05", "CA": "06",
    "CO": "08", "CT": "09", "DE": "10", "DC": "11", "FL": "12", "GA": "13",
    "HI": "15", "ID": "16", "IL": "17", "IN": "18", "IA": "19",
    "KS": "20", "KY": "21", "LA": "22", "ME": "23", "MD": "24",
    "MA": "25", "MI": "26", "MN": "27", "MS": "28", "MO": "29",
    "MT": "30", "NE": "31", "NV": "32", "NH": "33", "NJ": "34",
    "NM": "35", "NY": "36", "NC": "37", "ND": "38", "OH": "39",
    "OK": "40", "OR": "41", "PA": "42", "PR": "72", "RI": "44", "SC": "45",
    "SD": "46", "TN": "47", "TX": "48", "UT": "49", "VT": "50",
    "VA": "51", "VI": "78", "WA": "53", "WV": "54", "WI": "55", "WY": "56"
}

In [29]:
unique_origins.head()

,ORIGIN,ORIGIN_CITY
0,FLL,"Fort Lauderdale, FL"
1,MSP,"Minneapolis, MN"
2,DEN,"Denver, CO"
4,MCO,"Orlando, FL"
5,DAL,"Dallas, TX"


In [30]:
unique_origins[["CITY", "STATE"]] = unique_origins["ORIGIN_CITY"].str.rsplit(", ", n=1, expand=True)

print(unique_origins.head())


  ORIGIN          ORIGIN_CITY             CITY STATE
0    FLL  Fort Lauderdale, FL  Fort Lauderdale    FL
1    MSP      Minneapolis, MN      Minneapolis    MN
2    DEN           Denver, CO           Denver    CO
4    MCO          Orlando, FL          Orlando    FL
5    DAL           Dallas, TX           Dallas    TX


In [31]:
# Nhóm danh sách sân bay theo bang
state_airports = unique_origins.groupby("STATE")["ORIGIN"].apply(list).reset_index()

print(state_airports)


   STATE                                             ORIGIN
0     AK  [YAK, ANC, JNU, BRW, OTZ, WRG, SIT, KTN, CDV, ...
1     AL                     [HSV, BHM, MOB, MGM, DHN, BFM]
2     AR                               [XNA, FSM, LIT, TXK]
3     AZ                     [PRC, PHX, TUS, AZA, FLG, YUM]
4     CA  [SJC, SFO, LAX, SMF, SAN, FAT, OAK, SNA, SBP, ...
5     CO  [DEN, ASE, COS, EGE, GJT, MTJ, HDN, DRO, PUB, ...
6     CT                                         [BDL, HVN]
7     DC                                         [DCA, IAD]
8     DE                                              [ILG]
9     FL  [FLL, MCO, SRQ, TPA, MIA, RSW, MLB, VPS, PBI, ...
10    GA                [ATL, SAV, VLD, ABY, AGS, CSG, BQK]
11    HI                          [HNL, OGG, KOA, LIH, ITO]
12    IA                [CID, DSM, MCW, ALO, DBQ, SUX, FOD]
13    ID                     [BOI, PIH, IDA, SUN, LWS, TWF]
14    IL  [MDW, ORD, CMI, MLI, PIA, BMI, RFD, BLV, SPI, ...
15    IN                               [

In [32]:
import airportsdata

airports = airportsdata.load("IATA")

def get_airport_name(iata_code):
    return airports.get(iata_code, {}).get("name", "Unknown Airport")

unique_origins["AIRPORT_NAME"] = unique_origins["ORIGIN"].apply(get_airport_name)

print(unique_origins.head())

num_nulls = unique_origins["AIRPORT_NAME"].isna().sum()
print(f"Số lượng sân bay có giá trị NaN trong AIRPORT_NAME: {num_nulls}")


  ORIGIN          ORIGIN_CITY             CITY STATE  \
0    FLL  Fort Lauderdale, FL  Fort Lauderdale    FL   
1    MSP      Minneapolis, MN      Minneapolis    MN   
2    DEN           Denver, CO           Denver    CO   
4    MCO          Orlando, FL          Orlando    FL   
5    DAL           Dallas, TX           Dallas    TX   

                                        AIRPORT_NAME  
0    Fort Lauderdale/Hollywood International Airport  
1  Minneapolis-St Paul International/Wold-Chamber...  
2                       Denver International Airport  
4                      Orlando International Airport  
5                                  Dallas Love Field  
Số lượng sân bay có giá trị NaN trong AIRPORT_NAME: 0


In [ ]:
def get_noaa_stations_by_state(state_code):
    """Lấy danh sách trạm thời tiết NOAA cho một bang dựa trên mã FIPS"""
    if state_code not in state_fips:
        print(f"Không tìm thấy mã FIPS cho bang {state_code}")
        return []
    
    fips_code = state_fips[state_code]
    stations_url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/stations"
    all_stations = []
    offset = 1
    limit = 1000
    cutoff_date = datetime(2022, 1, 1)

    while True:
        params = {
            "datasetid": "GHCND",
            "locationid": f"FIPS:{fips_code}",
            "limit": limit,
            "offset": offset,
        }
        headers = {"token": API_KEY}

        response = requests.get(stations_url, headers=headers, params=params)

        if response.status_code != 200:
            print(f"Lỗi API khi lấy dữ liệu bang {state_code}! Mã lỗi: {response.status_code}")
            break

        try:
            response_json = response.json()
        except requests.exceptions.JSONDecodeError:
            print(f"Không thể decode JSON từ API cho bang {state_code}!")
            break

        if "results" in response_json:
            stations = response_json["results"]
            
            for s in stations:
                try:
                    max_date = datetime.strptime(s["maxdate"], "%Y-%m-%d")
                    if max_date >= cutoff_date:
                        all_stations.append({
                            "station_id": s["id"],
                            "name": s["name"],
                            "last_active": s["maxdate"],
                        })
                except KeyError:
                    print(f"Bỏ qua trạm {s['id']}")

            if len(stations) < limit:
                break 
            offset += limit
        else:
            print(f"Không tìm thấy trạm nào trong {state_code}!")
            break

    return all_stations

In [ ]:
fl_station = get_noaa_stations_by_state("TX")
print(f"Đã tìm thấy {len(fl_station)} trạm ở MN có dữ liệu từ 2022 trở đi!")

📍 Đã tìm thấy 3284 trạm ở MN có dữ liệu từ 2022 trở đi!


In [ ]:
# Danh sách lưu tất cả trạm thời tiết
all_weather_data = []

for state in state_airports["STATE"]:
    print(f"Đang lấy dữ liệu trạm thời tiết cho {state}...")
    stations = get_noaa_stations_by_state(state)
    
    if stations:
        for station in stations:
            station["STATE"] = state
            all_weather_data.append(station)

weather_stations_df = pd.DataFrame(all_weather_data)

print(weather_stations_df.head())

📡 Đang lấy dữ liệu trạm thời tiết cho AK...
📡 Đang lấy dữ liệu trạm thời tiết cho AL...
📡 Đang lấy dữ liệu trạm thời tiết cho AR...
📡 Đang lấy dữ liệu trạm thời tiết cho AZ...
📡 Đang lấy dữ liệu trạm thời tiết cho CA...
📡 Đang lấy dữ liệu trạm thời tiết cho CO...
📡 Đang lấy dữ liệu trạm thời tiết cho CT...
📡 Đang lấy dữ liệu trạm thời tiết cho DC...
📡 Đang lấy dữ liệu trạm thời tiết cho DE...
📡 Đang lấy dữ liệu trạm thời tiết cho FL...
📡 Đang lấy dữ liệu trạm thời tiết cho GA...
📡 Đang lấy dữ liệu trạm thời tiết cho HI...
📡 Đang lấy dữ liệu trạm thời tiết cho IA...
❌ Lỗi API khi lấy dữ liệu bang IA! Mã lỗi: 503
📡 Đang lấy dữ liệu trạm thời tiết cho ID...
📡 Đang lấy dữ liệu trạm thời tiết cho IL...
📡 Đang lấy dữ liệu trạm thời tiết cho IN...
📡 Đang lấy dữ liệu trạm thời tiết cho KS...
📡 Đang lấy dữ liệu trạm thời tiết cho KY...
📡 Đang lấy dữ liệu trạm thời tiết cho LA...
📡 Đang lấy dữ liệu trạm thời tiết cho MA...
📡 Đang lấy dữ liệu trạm thời tiết cho MD...
📡 Đang lấy dữ liệu trạm thời 

HÀM LẤY DỮ LIỆU TỪ TRẠM

In [39]:
weather_stations_df.shape

(36095, 4)

In [ ]:
# Xóa dữ liệu cũ của IA nếu có
all_weather_data = [s for s in all_weather_data if s["STATE"] != "IA"]

ia_station = get_noaa_stations_by_state("IA")
print(f"Đã tìm thấy {len(ia_station)} trạm ở IA có dữ liệu từ 2022 trở đi")

if ia_station:
    for station in ia_station:
        station["STATE"] = "IA"
        all_weather_data.append(station)

weather_stations_df = pd.DataFrame(all_weather_data)

print(weather_stations_df.head())

📍 Đã tìm thấy 606 trạm ở IA có dữ liệu từ 2022 trở đi!
          station_id                        name last_active STATE
0  GHCND:CA001206197        PLEASANT CAMP, AK CA  2022-05-04    AK
1  GHCND:US1AKAB0015    ANCHORAGE 5.0 ESE, AK US  2023-07-20    AK
2  GHCND:US1AKAB0021  EAGLE RIVER 2.6 ESE, AK US  2025-03-26    AK
3  GHCND:US1AKAB0038   EAGLE RIVER 7.8 SE, AK US  2025-03-20    AK
4  GHCND:US1AKAB0051      ANCHORAGE 4.5 E, AK US  2025-03-18    AK


In [42]:
weather_stations_df.shape

(36701, 4)

In [43]:
weather_stations_df.to_csv("weather_stations.csv", index=False, encoding="utf-8")

TH ĐẶC BIỆT

In [40]:
tt_airports = state_airports.loc[state_airports["STATE"] == "TT"]
print(tt_airports)

   STATE           ORIGIN
44    TT  [SPN, GUM, PPG]


In [ ]:
# Hàm tìm trạm thời tiết phù hợp với sân bay
def find_weather_stations_for_airport(airport_name, city, state, df_weather, top_n=3):
    # Lọc trạm thời tiết trong bang
    df_state_weather = df_weather[df_weather["STATE"] == state]
    
    # Tìm trạm có chứa tên sân bay
    name_match = df_state_weather[df_state_weather["name"].str.contains(airport_name, case=False, na=False)]
    if not name_match.empty:
        return name_match

    # Nếu không tìm thấy, chọn 3 trạm có chứa tên thành phố
    city_match = df_state_weather[df_state_weather["name"].str.contains(city, case=False, na=False)]
    if not city_match.empty:
        return city_match.head(top_n)

    return pd.DataFrame()

airport_weather_map = {}
skip_airports = {"SPN", "GUM", "PPG"}

for _, row in unique_origins.iterrows():
    iata_code = row["ORIGIN"]
    airport_name = row["AIRPORT_NAME"]
    city = row["CITY"]
    state = row["STATE"]
    
    if iata_code in skip_airports or state == "TT":
        print(f"Bỏ qua {iata_code}")
        continue
    
    matching_stations = find_weather_stations_for_airport(airport_name, city, state, weather_stations_df)
    airport_weather_map[iata_code] = matching_stations

# Chuyển thành DataFrame
if airport_weather_map:
    result_df = pd.concat(airport_weather_map.values(), keys=airport_weather_map.keys()).reset_index(level=0).rename(columns={"level_0": "IATA_CODE"})
    print(result_df.head())
else:
    result_df = pd.DataFrame()

C:\Users\Admin\AppData\Local\Temp\ipykernel_20872\743521838.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  name_match = df_state_weather[df_state_weather["name"].str.contains(airport_name, case=False, na=False)]
C:\Users\Admin\AppData\Local\Temp\ipykernel_20872\743521838.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  name_match = df_state_weather[df_state_weather["name"].str.contains(airport_name, case=False, na=False)]
C:\Users\Admin\AppData\Local\Temp\ipykernel_20872\743521838.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  name_match = df_state_weather[df_state_weather["name"].str.contains(airport_name, case=False, na=False)]
C:\Users\Admin\AppData\Local\Temp\ipykernel_20872\743521838.py:7: UserWarning: Thi

⏭️ Bỏ qua SPN do nằm ngoài lãnh thổ Mỹ.
⏭️ Bỏ qua GUM do nằm ngoài lãnh thổ Mỹ.
⏭️ Bỏ qua PPG do nằm ngoài lãnh thổ Mỹ.
      IATA_CODE         station_id                            name  \
6097        FLL  GHCND:US1FLBW0016  FORT LAUDERDALE 1.9 SSW, FL US   
6116        FLL  GHCND:US1FLBW0159  FORT LAUDERDALE 3.7 NNE, FL US   
6127        FLL  GHCND:US1FLBW0179  FORT LAUDERDALE 2.0 ENE, FL US   
14783       MSP  GHCND:US1MNHN0009      MINNEAPOLIS 3.0 NNW, MN US   
14788       MSP  GHCND:US1MNHN0022       MINNEAPOLIS 3.3 SW, MN US   

      last_active STATE  
6097   2022-07-19    FL  
6116   2025-03-26    FL  
6127   2025-02-24    FL  
14783  2022-04-25    MN  
14788  2025-03-25    MN  


In [50]:
result_df.to_csv("airport_weather.csv", index=False, encoding="utf-8")

In [12]:
wt_of_airport = pd.read_csv("airport_weather_station.csv")

In [ ]:
def get_weather_data_for_station(station_id, start_date, end_date):
    weather_url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"
    headers = {"token": API_KEY}
    
    all_data = []
    offset = 1
    limit = 1000
    max_retries = 3

    while True:
        params = {
            "datasetid": "GHCND",
            "stationid": station_id,
            "startdate": start_date,
            "enddate": end_date,
            "limit": limit,
            "offset": offset,
        }

        for attempt in range(max_retries):
            try:
                response = requests.get(weather_url, headers=headers, params=params)

                if response.status_code == 200:
                    data = response.json().get("results", [])
                    
                    if not data:
                        return all_data
                    
                    all_data.extend(data)
                    offset += limit
                    break

                elif response.status_code == 429:
                    print(f"Quá giới hạn request ({station_id}), đang thử lại...")
                    time.sleep(10)

                else:
                    print(f"Lỗi API ({station_id}): {response.status_code} - {response.text}")
                    return None

            except requests.exceptions.RequestException as e:
                print(f"Lỗi kết nối ({station_id}): {e}")

        else:
            print(f"Không thể lấy dữ liệu ({station_id}) sau {max_retries} lần thử.")
            return None 

    return all_data 

In [ ]:
response = get_weather_data_for_station("GHCND:US1FLBW0159", "2022-01-01", "2023-01-01")
print(response)

[{'date': '2022-01-01T00:00:00', 'datatype': 'PRCP', 'station': 'GHCND:US1FLBW0159', 'attributes': ',,N,0700', 'value': 0}, {'date': '2022-01-01T00:00:00', 'datatype': 'SNOW', 'station': 'GHCND:US1FLBW0159', 'attributes': ',,N,0700', 'value': 0}, {'date': '2022-01-02T00:00:00', 'datatype': 'PRCP', 'station': 'GHCND:US1FLBW0159', 'attributes': ',,N,0700', 'value': 0}, {'date': '2022-01-02T00:00:00', 'datatype': 'SNOW', 'station': 'GHCND:US1FLBW0159', 'attributes': ',,N,0700', 'value': 0}, {'date': '2022-01-03T00:00:00', 'datatype': 'PRCP', 'station': 'GHCND:US1FLBW0159', 'attributes': ',,N,0700', 'value': 0}, {'date': '2022-01-03T00:00:00', 'datatype': 'SNOW', 'station': 'GHCND:US1FLBW0159', 'attributes': ',,N,0700', 'value': 0}, {'date': '2022-01-04T00:00:00', 'datatype': 'PRCP', 'station': 'GHCND:US1FLBW0159', 'attributes': ',,N,0700', 'value': 5}, {'date': '2022-01-05T00:00:00', 'datatype': 'PRCP', 'station': 'GHCND:US1FLBW0159', 'attributes': ',,N,0700', 'value': 0}, {'date': '2022-

In [ ]:
if response:
    weather_df = pd.DataFrame(response)
    
    print("Kích thước dataframe:", weather_df.shape)
    
    print(weather_df.tail())
else:
    print("Không có dữ liệu")

📏 Kích thước DataFrame: (610, 5)
                    date datatype            station attributes  value
605  2022-12-30T00:00:00     PRCP  GHCND:US1FLBW0159   ,,N,0700     23
606  2022-12-31T00:00:00     PRCP  GHCND:US1FLBW0159   ,,N,0700      0
607  2022-12-31T00:00:00     SNOW  GHCND:US1FLBW0159   ,,N,0700      0
608  2023-01-01T00:00:00     PRCP  GHCND:US1FLBW0159   ,,N,0700      0
609  2023-01-01T00:00:00     SNOW  GHCND:US1FLBW0159   ,,N,0700      0


In [ ]:
# Thời gian lấy dữ liệu
START_DATE = "2022-01-01"
END_DATE = "2023-01-01"

weather_data = []

# Lặp qua từng trạm trong result_df
for _, row in result_df.iterrows():
    station_id = row["station_id"]
    iata_code = row["IATA_CODE"]

    print(f"Đang lấy dữ liệu từ trạm: {station_id} ({iata_code})...")

    # Gọi API để lấy dữ liệu
    data = get_weather_data_for_station(station_id, START_DATE, END_DATE)

    if data:
        for entry in data:
            weather_data.append({
                "IATA_CODE": iata_code,
                "station_id": station_id,
                "date": entry["date"],
                "datatype": entry["datatype"],
                "value": entry["value"]
            })

    time.sleep(1)

weather_df = pd.DataFrame(weather_data)

weather_df.to_csv("weather_data.csv", index=False)

print("Hoàn tất lấy dữ liệu thời tiết")
print(weather_df.head())

Đang lấy dữ liệu từ trạm: GHCND:US1FLBW0016 (FLL)...
Đang lấy dữ liệu từ trạm: GHCND:US1FLBW0159 (FLL)...
Đang lấy dữ liệu từ trạm: GHCND:US1FLBW0179 (FLL)...
Đang lấy dữ liệu từ trạm: GHCND:US1MNHN0009 (MSP)...
Đang lấy dữ liệu từ trạm: GHCND:US1MNHN0022 (MSP)...
Đang lấy dữ liệu từ trạm: GHCND:US1MNHN0028 (MSP)...
Đang lấy dữ liệu từ trạm: GHCND:USC00052212 (DEN)...
Đang lấy dữ liệu từ trạm: GHCND:USW00003017 (DEN)...
Đang lấy dữ liệu từ trạm: GHCND:USW00012815 (MCO)...
Đang lấy dữ liệu từ trạm: GHCND:US1TXDA0013 (DAL)...
Đang lấy dữ liệu từ trạm: GHCND:US1TXDA0048 (DAL)...
Đang lấy dữ liệu từ trạm: GHCND:US1TXDA0053 (DAL)...
Đang lấy dữ liệu từ trạm: GHCND:US1DCDC0009 (DCA)...
Đang lấy dữ liệu từ trạm: GHCND:US1DCDC0014 (DCA)...
Đang lấy dữ liệu từ trạm: GHCND:US1DCDC0026 (DCA)...
Đang lấy dữ liệu từ trạm: GHCND:US1ALMD0003 (HSV)...
Đang lấy dữ liệu từ trạm: GHCND:US1ALMD0012 (HSV)...
Đang lấy dữ liệu từ trạm: GHCND:US1ALMD0084 (HSV)...
Đang lấy dữ liệu từ trạm: GHCND:US1TXHRR147 (I

In [18]:
weather_df.shape

(1151688, 5)

In [ ]:
import pandas as pd

weather_df = pd.read_csv("weather_data.csv")

weather_pivot = weather_df.pivot_table(
    index=["IATA_CODE", "station_id", "date"],  # Các cột làm chỉ mục
    columns="datatype",
    values="value",
    aggfunc="sum"
).reset_index()

weather_pivot.columns.name = None  # Xóa tên index của cột

print(weather_pivot.head())


  IATA_CODE         station_id                 date   ADPT     ASLP     ASTP  \
0       ABE  GHCND:USW00014737  2022-01-01T00:00:00   94.0  10075.0   9942.0   
1       ABE  GHCND:USW00014737  2022-01-02T00:00:00   56.0  10064.0   9909.0   
2       ABE  GHCND:USW00014737  2022-01-03T00:00:00 -100.0  10210.0  10051.0   
3       ABE  GHCND:USW00014737  2022-01-04T00:00:00 -111.0  10284.0  10135.0   
4       ABE  GHCND:USW00014737  2022-01-05T00:00:00  -44.0  10129.0  10003.0   

   AWBT  AWND  DAPR  EVAP  ...  WT02  WT03  WT04  WT05  WT06  WT07  WT08  \
0  94.0  16.0   NaN   NaN  ...   1.0   NaN   NaN   NaN   NaN   NaN   NaN   
1  67.0  39.0   NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
2 -50.0  50.0   NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
3 -61.0  20.0   NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   1.0   
4 -22.0  11.0   NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   NaN   

   WT09  WT10  WT11  
0   NaN   NaN   NaN  
1   NaN   NaN   Na

In [5]:
print(weather_pivot.shape)

(185050, 44)


In [21]:
airports_from_origins = set(unique_origins["ORIGIN"])
airports_from_weather = set(weather_pivot["IATA_CODE"])

missing_airports = airports_from_origins - airports_from_weather

# Kiểm tra kết quả
if missing_airports:
    print("Các sân bay bị thiếu dữ liệu thời tiết:", missing_airports)
else:
    print("Tất cả sân bay trong unique_origins đều có dữ liệu thời tiết.")


Các sân bay bị thiếu dữ liệu thời tiết: {'LBE', 'RDM', 'PIT', 'RIW', 'TRI', 'PIR', 'PPG', 'YAK', 'DFW', 'SUN', 'JAN', 'SRQ', 'OAJ', 'ORH', 'ITH', 'CMX', 'JAC', 'DRT', 'AVP', 'LAW', 'MVY', 'MBS', 'GUM', 'GPT', 'MFE', 'HDN', 'CKB', 'MAF', 'BMI', 'BPT', 'ACV', 'MHK', 'CID', 'GNV', 'TUS', 'HOU', 'SPN', 'PSC', 'MDT', 'LGA', 'EKO', 'CMI', 'PIB', 'HPN', 'HTS', 'DLG', 'PHF', 'ELM', 'RDU', 'SWF', 'EVV', 'DDC'}


In [22]:
excluded_airports = {"SPN", "GUM", "PPG"}
missing_airports = missing_airports.difference(excluded_airports)

In [23]:
print(len(missing_airports))

49


In [ ]:
missing_stations = wt_of_airport[wt_of_airport["IATA_CODE"].isin(missing_airports)]
print(missing_stations)

    IATA_CODE         station_id  \
94        LGA  GHCND:USW00014732   
129       TUS  GHCND:USW00023160   
166       PIT  GHCND:USW00094823   
198       YAK  GHCND:USW00025339   
226       PIR  GHCND:USW00024025   
227       HOU  GHCND:USW00012918   
329       GNV  GHCND:USW00012816   
430       LBE  GHCND:US1PAWT0071   
431       LBE  GHCND:US1PAWT0079   
453       EVV  GHCND:USW00093817   
515       MDT  GHCND:USW00014711   
516       HDN  GHCND:USW00094025   
526       JAC  GHCND:USW00024166   
545       DDC  GHCND:USW00013985   
549       ORH  GHCND:USW00094746   
577       DRT  GHCND:USW00022010   
658       MBS  GHCND:USW00014845   
669       DLG  GHCND:USC00502457   
705       EKO  GHCND:USW00024121   

                                                  name last_active STATE  
94                            LAGUARDIA AIRPORT, NY US  2025-03-25    NY  
129                TUCSON INTERNATIONAL AIRPORT, AZ US  2025-03-25    AZ  
166            PITTSBURGH INTERNATIONAL AIRPORT, PA US

In [37]:
missing_stations.shape

(19, 5)

In [ ]:
START_DATE = "2022-01-01"
END_DATE = "2023-01-01"
new_weather_data = []

for _, row in missing_stations.iterrows():
    station_id = row["station_id"]
    iata_code = row["IATA_CODE"]

    print(f"Đang lấy dữ liệu từ trạm thiếu: {station_id} ({iata_code})...")

    data = get_weather_data_for_station(station_id, START_DATE, END_DATE)

    if data:
        for entry in data:
            new_weather_data.append({
                "IATA_CODE": iata_code,
                "station_id": station_id,
                "date": entry["date"],
                "datatype": entry["datatype"],
                "value": entry["value"]
            })

    time.sleep(1)

new_weather_df = pd.DataFrame(new_weather_data)

print(f"Dữ liệu mới thu thập: {new_weather_df.shape[0]} dòng")

📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00014732 (LGA)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00023160 (TUS)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00094823 (PIT)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00025339 (YAK)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00024025 (PIR)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00012918 (HOU)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00012816 (GNV)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:US1PAWT0071 (LBE)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:US1PAWT0079 (LBE)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00093817 (EVV)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00014711 (MDT)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00094025 (HDN)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00024166 (JAC)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00013985 (DDC)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00094746 (ORH)...
📡 Đang lấy dữ liệu từ trạm thiếu: GHCND:USW00022010 (DRT)...
📡 Đang lấy dữ liệu từ tr

In [20]:
updated_weather_df = pd.concat([weather_df, new_weather_df], ignore_index=True)
updated_weather_df.to_csv("updated_weather_data.csv", index=False)

In [7]:
updated_weather_df = pd.read_csv("updated_weather_data.csv")

In [ ]:
weather_pivot = updated_weather_df.pivot_table(
    index=["IATA_CODE", "station_id", "date"],  # Các cột làm chỉ mục
    columns="datatype",
    values="value",
    aggfunc="sum"
).reset_index()

weather_pivot.columns.name = None

print(weather_pivot.head())

  IATA_CODE         station_id                 date   ADPT     ASLP     ASTP  \
0       ABE  GHCND:USW00014737  2022-01-01T00:00:00   94.0  10075.0   9942.0   
1       ABE  GHCND:USW00014737  2022-01-02T00:00:00   56.0  10064.0   9909.0   
2       ABE  GHCND:USW00014737  2022-01-03T00:00:00 -100.0  10210.0  10051.0   
3       ABE  GHCND:USW00014737  2022-01-04T00:00:00 -111.0  10284.0  10135.0   
4       ABE  GHCND:USW00014737  2022-01-05T00:00:00  -44.0  10129.0  10003.0   

   AWBT  AWND  DAPR  EVAP  ...  WT02  WT03  WT04  WT05  WT06  WT07  WT08  \
0  94.0  16.0   NaN   NaN  ...   1.0   NaN   NaN   NaN   NaN   NaN   NaN   
1  67.0  39.0   NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
2 -50.0  50.0   NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
3 -61.0  20.0   NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   1.0   
4 -22.0  11.0   NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   NaN   

   WT09  WT10  WT11  
0   NaN   NaN   NaN  
1   NaN   NaN   Na

In [9]:
weather_pivot.shape

(190871, 44)

In [ ]:
datatypes = list(weather_pivot.columns[3:])
print(datatypes)

['ADPT', 'ASLP', 'ASTP', 'AWBT', 'AWND', 'DAPR', 'EVAP', 'MDPR', 'MNPN', 'MXPN', 'PGTM', 'PRCP', 'PSUN', 'RHAV', 'RHMN', 'RHMX', 'SNOW', 'SNWD', 'TAVG', 'TMAX', 'TMIN', 'TOBS', 'TSUN', 'WDF2', 'WDF5', 'WESD', 'WESF', 'WSF2', 'WSF5', 'WSFG', 'WT01', 'WT02', 'WT03', 'WT04', 'WT05', 'WT06', 'WT07', 'WT08', 'WT09', 'WT10', 'WT11']


In [ ]:
columns_to_keep = [
    "IATA_CODE", "station_id", "date",
    "TAVG", "TMAX", "TMIN",  # Nhiệt độ
    "AWND", "WSF2", "WSF5", "WSFG",  # Gió
    "PRCP", "SNOW", "SNWD",  # Giáng thủy
    "WT01", "WT02", "WT03", "WT04", "WT05", "WT06", "WT07", "WT09", "WT10", "WT11"  # Thời tiết cực đoan
]

filtered_weather = weather_pivot[columns_to_keep]

print(filtered_weather.head())

  IATA_CODE         station_id                 date  TAVG   TMAX  TMIN  AWND  \
0       ABE  GHCND:USW00014737  2022-01-01T00:00:00   NaN  117.0  89.0  16.0   
1       ABE  GHCND:USW00014737  2022-01-02T00:00:00   NaN  139.0  -5.0  39.0   
2       ABE  GHCND:USW00014737  2022-01-03T00:00:00   NaN    0.0 -77.0  50.0   
3       ABE  GHCND:USW00014737  2022-01-04T00:00:00   NaN   11.0 -88.0  20.0   
4       ABE  GHCND:USW00014737  2022-01-05T00:00:00   NaN   44.0 -55.0  11.0   

   WSF2   WSF5  WSFG  ...  WT01  WT02  WT03  WT04  WT05  WT06  WT07  WT09  \
0  45.0   54.0   NaN  ...   1.0   1.0   NaN   NaN   NaN   NaN   NaN   NaN   
1  89.0  112.0   NaN  ...   1.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
2  94.0  116.0   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
3  63.0   72.0   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
4  40.0   58.0   NaN  ...   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   

   WT10  WT11  
0   NaN   NaN  
1   NaN   NaN  
2   NaN 

In [3]:
df_filtered = df[[
    "FL_DATE",
    "AIRLINE",
    "AIRLINE_CODE",    
    "DOT_CODE",        
    "FL_NUMBER",       
    "ORIGIN",         
    "DEST",           
    "CRS_DEP_TIME",    
    "CRS_ARR_TIME",    
    "CRS_ELAPSED_TIME",
    "DISTANCE",        
    "CANCELLED",       
    "DEP_DELAY"        
]]

In [4]:
df_filtered = df_filtered.copy()

df_filtered.loc[:, "FL_DATE"] = pd.to_datetime(df_filtered["FL_DATE"])

start_date = pd.to_datetime("2022-01-01")
end_date = pd.to_datetime("2023-01-01")

df_filtered = df_filtered[(df_filtered["FL_DATE"] >= start_date) & (df_filtered["FL_DATE"] < end_date)]

In [5]:
df_filtered.shape

(687860, 13)

In [11]:
valid_airports = set(updated_weather_df["IATA_CODE"].unique())

df_filtered = df_filtered[(df_filtered["ORIGIN"].isin(valid_airports)) & 
                          (df_filtered["DEST"].isin(valid_airports))]


In [12]:
df_filtered.tail(20)

,FL_DATE,AIRLINE,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,CANCELLED,DEP_DELAY
2999906,2022-10-06 00:00:00,PSA Airlines Inc.,OH,20397,5502,CLT,ABE,1719,1900,101.0,481.0,0.0,-5.0
2999907,2022-09-27 00:00:00,Southwest Airlines Co.,WN,19393,1416,CVG,BWI,610,740,90.0,430.0,0.0,1.0
2999908,2022-03-04 00:00:00,United Air Lines Inc.,UA,19977,281,CLT,IAD,800,926,86.0,322.0,0.0,6.0
2999912,2022-02-15 00:00:00,Southwest Airlines Co.,WN,19393,1408,ATL,MCI,1935,2050,135.0,692.0,0.0,-1.0
2999917,2022-11-27 00:00:00,Southwest Airlines Co.,WN,19393,1799,BNA,MCO,500,755,115.0,616.0,0.0,-5.0
2999923,2022-02-03 00:00:00,United Air Lines Inc.,UA,19977,1711,MSN,ORD,855,958,63.0,109.0,0.0,-1.0
2999931,2022-09-21 00:00:00,Delta Air Lines Inc.,DL,19790,1294,LAX,MSP,2335,506,211.0,1535.0,0.0,-2.0
2999940,2022-04-18 00:00:00,SkyWest Airlines Inc.,OO,20304,3241,ORD,MKE,2015,2103,48.0,67.0,0.0,-2.0
2999942,2022-09-19 00:00:00,Republic Airline,YX,20452,4561,MSY,DCA,1446,1822,156.0,969.0,0.0,23.0
2999952,2022-06-08 00:00:00,JetBlue Airways,B6,20409,2986,MIA,LAX,806,1053,347.0,2342.0,0.0,-8.0


CHUẨN HÓA DỮ LIỆU THỜI TIẾT

In [6]:
filtered_weather["date"] = pd.to_datetime(filtered_weather["date"])

filtered_weather["TAVG"] = filtered_weather["TAVG"].fillna((filtered_weather["TMAX"] + filtered_weather["TMIN"]) / 2)
filtered_weather["TMAX"] = filtered_weather["TMAX"] / 10.0
filtered_weather["TMIN"] = filtered_weather["TMIN"] / 10.0
filtered_weather["TAVG"] = filtered_weather["TAVG"] / 10.0

temp_cols = ["TAVG", "TMAX", "TMIN"]
filtered_weather[temp_cols] = filtered_weather[temp_cols].fillna(filtered_weather[temp_cols].mean())

wind_cols = ["AWND", "WSF2", "WSF5", "WSFG"]
for col in wind_cols:
    filtered_weather[col] = filtered_weather[col] / 10.0

for col in wind_cols:
    filtered_weather[col].fillna(filtered_weather[col].median(), inplace=True)

filtered_weather["PRCP"].fillna(0, inplace=True)
filtered_weather["SNOW"].fillna(0, inplace=True) 
filtered_weather["SNWD"].fillna(0, inplace=True) 

weather_extreme_cols = ["WT01", "WT02", "WT03", "WT04", "WT05", "WT06", "WT07", "WT09", "WT10", "WT11"]
filtered_weather[weather_extreme_cols] = filtered_weather[weather_extreme_cols].fillna(0)

print(filtered_weather.head())

  IATA_CODE         station_id       date   TAVG  TMAX  TMIN  AWND  WSF2  \
0       ABE  GHCND:USW00014737 2022-01-01  10.30  11.7   8.9   1.6   4.5   
1       ABE  GHCND:USW00014737 2022-01-02   6.70  13.9  -0.5   3.9   8.9   
2       ABE  GHCND:USW00014737 2022-01-03  -3.85   0.0  -7.7   5.0   9.4   
3       ABE  GHCND:USW00014737 2022-01-04  -3.85   1.1  -8.8   2.0   6.3   
4       ABE  GHCND:USW00014737 2022-01-05  -0.55   4.4  -5.5   1.1   4.0   

   WSF5  WSFG  ...  WT01  WT02  WT03  WT04  WT05  WT06  WT07  WT09  WT10  WT11  
0   5.4   4.5  ...   1.0   1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
1  11.2   4.5  ...   1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
2  11.6   4.5  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
3   7.2   4.5  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
4   5.8   4.5  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  

[5 rows x 23 columns]


C:\Users\Admin\AppData\Local\Temp\ipykernel_26296\1697607696.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_weather["date"] = pd.to_datetime(filtered_weather["date"])
C:\Users\Admin\AppData\Local\Temp\ipykernel_26296\1697607696.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_weather["TAVG"] = filtered_weather["TAVG"].fillna((filtered_weather["TMAX"] + filtered_weather["TMIN"]) / 2)
C:\Users\Admin\AppData\Local\Temp\ipykernel_26296\1697607696.py:4: SettingWithCopyWarning: 
A value 

In [7]:
print(filtered_weather.isna().any().any())
print(filtered_weather[filtered_weather.isna().any(axis=1)])

False
Empty DataFrame
Columns: [IATA_CODE, station_id, date, TAVG, TMAX, TMIN, AWND, WSF2, WSF5, WSFG, PRCP, SNOW, SNWD, WT01, WT02, WT03, WT04, WT05, WT06, WT07, WT09, WT10, WT11]
Index: []

[0 rows x 23 columns]


In [8]:
filtered_weather.to_csv("no_na_weather.csv", index=False)

In [ ]:
import airportsdata
airports = airportsdata.load("IATA")

def get_airport_name(iata_code):
    return airports.get(iata_code, {}).get("name", "Unknown Airport")

df_filtered["AIRPORT_NAME"] = df_filtered["ORIGIN"].apply(get_airport_name)

print(df_filtered.head())

num_nulls = df_filtered["AIRPORT_NAME"].isna().sum()
print(f"Số lượng sân bay có giá trị NaN trong AIRPORT_NAME: {num_nulls}")


                FL_DATE                 AIRLINE AIRLINE_CODE  DOT_CODE  \
1   2022-11-19 00:00:00    Delta Air Lines Inc.           DL     19790   
2   2022-07-22 00:00:00   United Air Lines Inc.           UA     19977   
15  2022-05-01 00:00:00  Southwest Airlines Co.           WN     19393   
20  2022-05-05 00:00:00         JetBlue Airways           B6     20409   
22  2022-11-12 00:00:00    Delta Air Lines Inc.           DL     19790   

    FL_NUMBER ORIGIN DEST  CRS_DEP_TIME  CRS_ARR_TIME  CRS_ELAPSED_TIME  \
1        1149    MSP  SEA          2120          2315             235.0   
2         459    DEN  MSP           954          1252             118.0   
15       1011    BWI  BDL          1735          1840              65.0   
20       1273    JFK  CHS           803          1012             129.0   
22       2706    GRR  MSP           730           806              96.0   

    DISTANCE  CANCELLED  DEP_DELAY  \
1     1399.0        0.0       -6.0   
2      680.0        0.0     

In [14]:
df_filtered.to_csv("filtered_flight.csv", index=False)